# turkish_distill: TR-MTEB evaluation

Evaluates any embedding model (teachers, baselines, our trained students) on TR-MTEB (`MTEB(Turkish)`, 26 tasks) and compares the finished runs. This notebook is only for evaluation; training lives in `colab_pipeline.ipynb`.

- Results go to Drive: `bitirme/eval/<label>/` (one folder per model, one JSON per task). An interrupted run resumes and skips finished tasks.
- Models are downloaded to the Colab local disk, never to Drive.
- Default protocol: plain encoding, **no instruction/prefix** (this is how the TR-MTEB authors evaluate). Prefixed variants are separate presets with their own labels.
- Use a **separate runtime** from the one doing teacher encoding: both need the GPU.

Before the first run: `HF_TOKEN` in Colab Secrets.

## 1. Setup

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'
EVAL_DIR = f'{DRIVE_ROOT}/eval'

os.makedirs(EVAL_DIR, exist_ok=True)
# HF cache on the LOCAL disk (Drive corrupts large model files).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
# The TR-MTEB authors' fork of MTEB. It pins datasets<3, so pip may downgrade `datasets` in this runtime.
!pip install --quiet "git+https://github.com/selmanbaysan/mteb_tr.git"
!git log -1 --oneline

Check that the Turkish benchmark is available. If this fails, restart the runtime once, then **re-run the Setup cell above again** (a restart clears all variables such as `DRIVE_ROOT` and `EVAL_DIR`) before continuing:

In [ ]:
import mteb
print('mteb', getattr(mteb, '__version__', '?'))
bench = mteb.get_benchmark('MTEB(Turkish)')
print(len(list(bench.tasks)), 'tasks')

## 2. Model presets

Revisions are Hugging Face commit hashes fetched on 2026-10-11, so every run is reproducible. A preset is just the arguments of `run_evaluation`; add your own entries (e.g. a trained student: `model` = path of its `final/` folder on Drive).

Reference numbers for a sanity check (other papers/cards, their own protocols, so only roughly comparable): Qwen3-Embedding-8B 68.66, mE5-large-instruct 67.47, BGE-M3 67.36 (MoganBert paper, TR-MTEB mean over tasks); `modernbert-tr-embed` 68.14 and `turkish-e5-large` 67.17 (model card). A plain run that is far below these probably means a protocol difference (see the `*_instr` presets).

In [ ]:
INSTR = 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: '

# Prompt-sensitive models are scored twice: plain (the TR-MTEB authors' protocol) and with their
# query/document prefixes (`*_instr`). Prompts only change RETRIEVAL scores (verified on mE5: the 15 other
# tasks are bit-identical), so every `*_instr` preset has `reuse_from=<its plain preset>`: it copies the
# plain run's non-retrieval results and only evaluates the 11 retrieval tasks. Run the plain preset first.
# Only the EmbeddingGemma prefixes below were checked against its model card; the Qwen3 instruction was
# written from memory. Models without an `_instr` preset are run plain only, so their retrieval score may
# understate what they reach with the prefixes they expect.
PRESETS = {
    # --- teacher candidates ---
    'bge_m3': dict(model='BAAI/bge-m3', revision='5617a9f61b028005a4858fdac845db406aefb181', dtype='float16'),
    'me5_large_instruct': dict(model='intfloat/multilingual-e5-large-instruct', revision='274baa43b0e13e37fafa6428dbc7938e62e5c439', dtype='float16'),
    'me5_large_instruct_instr': dict(model='intfloat/multilingual-e5-large-instruct', revision='274baa43b0e13e37fafa6428dbc7938e62e5c439', dtype='float16', query_prompt=INSTR,
                                     reuse_from='me5_large_instruct'),
    'qwen3_8b': dict(model='Qwen/Qwen3-Embedding-8B', revision='1d8ad4ca9b3dd8059ad90a75d4983776a23d44af', dtype='bfloat16', batch_size=16),
    'qwen3_8b_instr': dict(model='Qwen/Qwen3-Embedding-8B', revision='1d8ad4ca9b3dd8059ad90a75d4983776a23d44af', dtype='bfloat16', batch_size=16, query_prompt=INSTR,
                           reuse_from='qwen3_8b'),
    'turkish_e5_large': dict(model='ytu-ce-cosmos/turkish-e5-large', revision='02e2362d503bbdeafcb17143b2165c0743f9fdb1', dtype='float16'),

    # --- more teacher candidates (added 2026-10-11) ---
    # Gated: accept the license on the model page with the same Hugging Face account as HF_TOKEN.
    'embeddinggemma_300m': dict(model='google/embeddinggemma-300m', revision='57c266a740f537b4dc058e1b0cda161fd15afa75', dtype='bfloat16'),
    'embeddinggemma_300m_instr': dict(model='google/embeddinggemma-300m', revision='57c266a740f537b4dc058e1b0cda161fd15afa75', dtype='bfloat16',
                                      query_prompt='task: search result | query: ', passage_prompt='title: none | text: ',
                                      reuse_from='embeddinggemma_300m'),
    'harrier_0_6b': dict(model='microsoft/harrier-oss-v1-0.6b', revision='f9b9dc8d367d443f2479d27aa5d8d2850c0774ee', dtype='bfloat16'),
    'harrier_270m': dict(model='microsoft/harrier-oss-v1-270m', revision='31de22b673913c7d658c0f03f792d77c2dcf8ebd', dtype='bfloat16'),
    'gte_multilingual_base': dict(model='Alibaba-NLP/gte-multilingual-base', revision='9bbca17d9273fd0d03d5725c7a4b0f6b45142062', dtype='float16', trust_remote_code=True),
    'arctic_embed_l_v2': dict(model='Snowflake/snowflake-arctic-embed-l-v2.0', revision='ac6544c8a46e00af67e330e85a9028c66b8cfd9a', dtype='float16'),
    'nomic_embed_v2_moe': dict(model='nomic-ai/nomic-embed-text-v2-moe', revision='1066b6599d099fbb93dfcb64f9c37a7c9e503e85', dtype='float32', trust_remote_code=True),
    'qwen3_4b': dict(model='Qwen/Qwen3-Embedding-4B', revision='5cf2132abc99cad020ac570b19d031efec650f2b', dtype='bfloat16', batch_size=16),
    'qwen3_4b_instr': dict(model='Qwen/Qwen3-Embedding-4B', revision='5cf2132abc99cad020ac570b19d031efec650f2b', dtype='bfloat16', batch_size=16, query_prompt=INSTR,
                           reuse_from='qwen3_4b'),
    'qwen3_0_6b': dict(model='Qwen/Qwen3-Embedding-0.6B', revision='97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3', dtype='bfloat16'),

    # --- baselines from the same lab ---
    'modernbert_tr_embed': dict(model='ytu-ce-cosmos/modernbert-tr-embed', revision='855820e0079102805908d60d32cd9fd59c970b80'),
    'modernbert_tr_1k_embed': dict(model='ytu-ce-cosmos/modernbert-tr-1k-embed', revision='ba1e17bc7e95a257b4e614c300c4ca076711fce6'),
    # --- our student before any distillation (raw MLM encoder + mean pooling) ---
    'modernbert_tr_base_1k_raw': dict(model='ytu-ce-cosmos/modernbert-tr-base-1k', revision='c2e35f553101cce5b03fbb318ef8728473fd34dd'),
    # --- our trained students: point `model` at a run's final/ folder ---
    # 'my_student': dict(model=f'{DRIVE_ROOT}/runs/multi_a/<run_id>/final'),
}
print(list(PRESETS))

## 3. Run one evaluation

`TASKS = None` runs all 26 tasks. For a quick smoke test use a few small tasks; they are stored under a separate label (`<name>__subset`) so they never mix with full runs.

In [ ]:
CHOICE = 'bge_m3'
TASKS = None   # e.g. ['STSbTR', 'TurkishIronyClassification', 'TurkishMovieSentimentClassification']

# Guard: this cell needs the Setup cell (section 1) to have run in THIS runtime session
# (it defines EVAL_DIR and PRESETS comes from section 2). If the package is missing but the
# repo is there, install it now instead of failing with ModuleNotFoundError.
import importlib.util, os, subprocess, sys
REPO_DIR = globals().get('REPO_DIR', '/content/repo')
if importlib.util.find_spec('turkish_distill') is None:
    if not os.path.isdir(f'{REPO_DIR}/src'):
        raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')

from turkish_distill.eval.trmteb import run_evaluation, summarize_dir, format_table

label = CHOICE if TASKS is None else f'{CHOICE}__subset'
label_dir = run_evaluation(output_dir=EVAL_DIR, label=label, task_names=TASKS, **PRESETS[CHOICE])
print(format_table({label: summarize_dir(label_dir)}))

In [ ]:
# Run several presets one after another (e.g. before going away). Each model runs in its OWN Python process:
# if one fails (gated access, incompatible remote code, a CUDA "device-side assert"...) the error is reported
# and the next model starts in a clean process. An interrupted preset resumes: finished tasks are skipped.
# Put the plain preset before its `*_instr` variant (the variant reuses the plain run's 15 non-retrieval results).
#
# Already done (2026-10-11): bge_m3, me5_large_instruct(+_instr), qwen3_8b, turkish_e5_large, embeddinggemma_300m(+_instr),
# harrier_0_6b, harrier_270m, modernbert_tr_embed, modernbert_tr_1k_embed, modernbert_tr_base_1k_raw.
# Known incompatible with the current transformers: gte_multilingual_base (its remote code calls a removed method).
BATCH = ['arctic_embed_l_v2', 'qwen3_0_6b', 'nomic_embed_v2_moe']   # the remote-code model goes last

import time
from turkish_distill.eval.trmteb import run_evaluation_isolated, summarize_dir, format_table

failed = {}
for name in BATCH:
    t0 = time.time()
    print(f'\n########## {name}')
    try:
        d = run_evaluation_isolated(label=name, output_dir=EVAL_DIR, **PRESETS[name])
        print(f'\n=== {name} done in {(time.time() - t0) / 60:.1f} min')
        print(format_table({name: summarize_dir(d)}))
    except Exception as exc:
        failed[name] = f'{type(exc).__name__}: {str(exc)[:300]}'
        print(f'\n=== {name} FAILED after {(time.time() - t0) / 60:.1f} min: {failed[name]}')

print('\nfailed:', failed or 'none')

## 4. Compare all finished evaluations

In [ ]:
import glob
import pandas as pd

dirs = sorted(d for d in glob.glob(f'{EVAL_DIR}/*') if os.path.isdir(d) and not d.endswith('__subset'))
summaries = {os.path.basename(d): summarize_dir(d) for d in dirs}
print(format_table(summaries))

per_task = pd.DataFrame({k: v['per_task'] for k, v in summaries.items()}).round(2)
per_task